# English ↔ Nepali Machine Translation
This notebook trains a sequence-to-sequence translation model (English ↔ Nepali) using the `CohleM/english-to-nepali` dataset from Hugging Face. It implements two attention mechanisms (General and Additive), compares them, visualizes attention maps, and includes a minimal Flask app example for serving translations.

References: dataset from https://huggingface.co/datasets/CohleM/english-to-nepali (credit: CohleM). SentencePiece tokenizer by Google (https://github.com/google/sentencepiece).

**Tasks covered**:
- Task 1: Dataset selection & preprocessing (credit and steps).
- Task 2: Implement General and Additive attention in an LSTM seq2seq.
- Task 3: Evaluation (BLEU), training/validation plots, attention maps.
- Task 4: Simple web application example to serve the trained model.

## 1) Dataset and preprocessing (Task 1)
**Dataset**: `CohleM/english-to-nepali` from Hugging Face Datasets. Credit: CohleM, hosted on Hugging Face. URL: https://huggingface.co/datasets/CohleM/english-to-nepali

**Preprocessing steps**:
1. Load dataset via `datasets.load_dataset`.
2. Normalize punctuation and Unicode NFC.
3. Train a joint SentencePiece model (subword) on combined English+Nepali text to handle morphological richness and rare words. SentencePiece is robust for Devanagari scripts (Nepali uses Devanagari). Credit: Google SentencePiece (https://github.com/google/sentencepiece).
4. Encode sentences into token ids, add BOS/EOS tokens, and create PyTorch datasets.

**Tools & libraries**: `datasets` (Hugging Face), `sentencepiece`, `tokenizers` or `transformers`-style processing, `torch` for models. Credit: Hugging Face team for `datasets`, PyTorch team for `torch`.

In [3]:
# Install required packages if not already installed (uncomment to run)
# !pip install datasets sentencepiece sacrebleu matplotlib seaborn torch torchvision tqdm flask

import os
import math
import random
from pathlib import Path
import sentencepiece as spm
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import sacrebleu
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [4]:
# Load the Hugging Face dataset
ds = load_dataset('CohleM/english-to-nepali')
ds

README.md:   0%|          | 0.00/328 [00:00<?, ?B/s]

c:\Users\gaurav\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gaurav\.cache\huggingface\hub\datasets--CohleM--english-to-nepali. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00001-ea91b3ffe28041(…):   0%|          | 0.00/28.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/177334 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['en', 'ne'],
        num_rows: 177334
    })
})

In [5]:
# Prepare data files for SentencePiece training (combine train split)
train_en = [ex['en'] for ex in ds['train']]
train_ne = [ex['ne'] for ex in ds['train']]
len(train_en), len(train_ne)

(177334, 177334)

In [ ]:
# Write combined corpus to a temporary file for SentencePiece
sp_corpus = Path('sp_corpus.txt')
with sp_corpus.open('w', encoding='utf-8') as f:
    for s in train_en:
        f.write(s.replace('\n', ' ') + '\n')
    for s in train_ne:
        f.write(s.replace('\n', ' ') + '\n')

# Train SentencePiece model
vocab_size = 8000
sp_model_prefix = 'spm_en_ne'
if not Path(sp_model_prefix + '.model').exists():
    spm.SentencePieceTrainer.Train(f'--input={str(sp_corpus)} --model_prefix={sp_model_prefix} --vocab_size={vocab_size} --character_coverage=1.0 --model_type=bpe')

sp = spm.SentencePieceProcessor()
sp.load(sp_model_prefix + '.model')
print('SentencePiece vocab size:', sp.get_piece_size())

SyntaxError: unterminated string literal (detected at line 5) (854522707.py, line 5)

In [8]:
# Tokenization helpers using SentencePiece
BOS = '<s>'
EOS = '</s>'
PAD = '<pad>'
UNK = '<unk>'

bos_id = sp.piece_to_id(BOS) if sp.is_unknown(bos_id:=0) else sp.piece_to_id(BOS) if BOS in sp else None
# We'll set special ids manually by adding if missing
# For simplicity, use the raw ids from sentencepiece and rely on reserved tokens later.

def encode(text, add_bos_eos=True, max_len=128):
    ids = sp.encode(text, out_type=int)
    if add_bos_eos:
        ids = [sp.bos_id() if hasattr(sp, 'bos_id') else sp.piece_to_id(BOS) if BOS in sp else sp.piece_to_id(EOS)] + ids + [sp.eos_id() if hasattr(sp, 'eos_id') else sp.piece_to_id(EOS) if EOS in sp else sp.piece_to_id(EOS)]
    if len(ids) > max_len:
        ids = ids[:max_len]
    return ids

def decode(ids):
    return sp.decode(ids)

NameError: name 'sp' is not defined

In [ ]:
# Build PyTorch Dataset
class MTDataset(Dataset):
    def __init__(self, examples, src_key='en', tgt_key='ne', max_len=64):
        self.examples = examples
        self.src_key = src_key
        self.tgt_key = tgt_key
        self.max_len = max_len
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        src_ids = sp.encode(ex[self.src_key], out_type=int)[:self.max_len-2]
        tgt_ids = sp.encode(ex[self.tgt_key], out_type=int)[:self.max_len-2]
        # add bos/eos using sentencepiece ids if available
        src = [sp.piece_to_id(BOS) if BOS in sp else sp.bos_id()] + src_ids + [sp.piece_to_id(EOS) if EOS in sp else sp.eos_id()]
        tgt = [sp.piece_to_id(BOS) if BOS in sp else sp.bos_id()] + tgt_ids + [sp.piece_to_id(EOS) if EOS in sp else sp.eos_id()]
        return torch.tensor(src, dtype=torch.long), torch.tensor(tgt, dtype=torch.long)

def collate_fn(batch):
    srcs, tgts = zip(*batch)
    src_lens = [len(s) for s in srcs]
    tgt_lens = [len(t) for t in tgts]
    src_pad = nn.utils.rnn.pad_sequence(srcs, padding_value=sp.piece_to_id(PAD) if PAD in sp else 0, batch_first=True)
    tgt_pad = nn.utils.rnn.pad_sequence(tgts, padding_value=sp.piece_to_id(PAD) if PAD in sp else 0, batch_first=True)
    return src_pad, tgt_pad, torch.tensor(src_lens), torch.tensor(tgt_lens)

# Create train/validation datasets
train_ds = MTDataset(list(ds['train']))
# If dataset has validation split, use it otherwise split train
if 'validation' in ds:
    val_ds = MTDataset(list(ds['validation']))
else:
    n = len(train_ds)
    cut = int(0.9 * n)
    val_ds = MTDataset(list(ds['train'])[cut:])
    train_ds = MTDataset(list(ds['train'])[:cut])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)
print('Train size:', len(train_ds), 'Val size:', len(val_ds))

## 2) Model: Encoder-Decoder with Attention (General & Additive)
We'll implement a BiLSTM encoder and an LSTM decoder with a selectable attention mechanism. The attention modules implement the equations: 
- General: e_i = s^T W h_i  (implemented as a linear layer mapping encoder hidden to decoder dim)
- Additive: e_i = v^T tanh(W1 h_i + W2 s)

In [ ]:
class GeneralAttention(nn.Module):
    def __init__(self, enc_hid, dec_hid):
        super().__init__()
        self.W = nn.Linear(enc_hid, dec_hid, bias=False)
    def forward(self, decoder_state, encoder_outputs, mask=None):
        # encoder_outputs: (B, T, enc_hid)
        # decoder_state: (B, dec_hid)
        proj = self.W(encoder_outputs)  # (B, T, dec_hid)
        # score = s^T (W h_i) -> dot product along dec_hid
        scores = torch.bmm(proj, decoder_state.unsqueeze(2)).squeeze(2)  # (B, T)
        if mask is not None:
            scores = scores.masked_fill(mask==0, -1e9)
        attn = torch.softmax(scores, dim=1)  # (B, T)
        context = torch.bmm(attn.unsqueeze(1), encoder_outputs).squeeze(1)  # (B, enc_hid)
        return context, attn

class AdditiveAttention(nn.Module):
    def __init__(self, enc_hid, dec_hid):
        super().__init__()
        self.W1 = nn.Linear(enc_hid, dec_hid, bias=False)
        self.W2 = nn.Linear(dec_hid, dec_hid, bias=False)
        self.v = nn.Linear(dec_hid, 1, bias=False)
    def forward(self, decoder_state, encoder_outputs, mask=None):
        # encoder_outputs: (B, T, enc_hid)
        # decoder_state: (B, dec_hid)
        enc_proj = self.W1(encoder_outputs)  # (B, T, dec_hid)
        dec_proj = self.W2(decoder_state).unsqueeze(1)  # (B, 1, dec_hid)
        score = self.v(torch.tanh(enc_proj + dec_proj)).squeeze(2)  # (B, T)
        if mask is not None:
            score = score.masked_fill(mask==0, -1e9)
        attn = torch.softmax(score, dim=1)
        context = torch.bmm(attn.unsqueeze(1), encoder_outputs).squeeze(1)
        return context, attn

class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hid, n_layers=1, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.rnn = nn.LSTM(emb_dim, enc_hid//2, num_layers=n_layers, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(dropout)
    def forward(self, src, src_len):
        # src: (B, T)
        embedded = self.dropout(self.embedding(src))
        packed = nn.utils.rnn.pack_padded_sequence(embedded, src_len.cpu(), batch_first=True, enforce_sorted=False)
        outputs, (h, c) = self.rnn(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(outputs, batch_first=True)
        # outputs: (B, T, enc_hid)
        return outputs, (h, c)

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hid, dec_hid, attention, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.rnn = nn.LSTM(emb_dim + enc_hid, dec_hid, batch_first=True)
        self.attention = attention
        self.fc_out = nn.Linear(dec_hid + enc_hid + emb_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)
    def forward(self, input_step, last_hidden, encoder_outputs, mask=None):
        # input_step: (B) - token ids for current time step
        embedded = self.dropout(self.embedding(input_step).unsqueeze(1))  # (B,1,emb_dim)
        decoder_state = last_hidden[0][-1]  # (B, dec_hid) - top layer hidden
        context, attn = self.attention(decoder_state, encoder_outputs, mask=mask)
        rnn_input = torch.cat([embedded, context.unsqueeze(1)], dim=2)
        output, hidden = self.rnn(rnn_input, last_hidden)
        output = output.squeeze(1)  # (B, dec_hid)
        concat = torch.cat([output, context, embedded.squeeze(1)], dim=1)
        preds = self.fc_out(concat)
        return preds, hidden, attn

# Utility to create mask from lengths
def make_src_mask(src, src_len):
    B, T = src.shape
    mask = torch.arange(T, device=src.device).unsqueeze(0) < src_len.unsqueeze(1)
    return mask  # (B, T)

In [ ]:
# Training / evaluation loop (teacher forcing)
def train_epoch(encoder, decoder, data_loader, enc_optimizer, dec_optimizer, criterion, clip=1.0, teacher_forcing_ratio=0.5):
    encoder.train(); decoder.train()
    total_loss = 0
    for src, tgt, src_lens, tgt_lens in tqdm(data_loader):
        src, tgt = src.to(device), tgt.to(device)
        src_mask = make_src_mask(src, src_lens).to(device)
        enc_optimizer.zero_grad(); dec_optimizer.zero_grad()
        encoder_outputs, enc_hidden = encoder(src, src_lens)
        # initialize decoder hidden from encoder final states (simple transform)
        # For simplicity, create zeros hidden of proper size
        batch_size = src.size(0)
        dec_hidden = (torch.zeros(1, batch_size, decoder.rnn.hidden_size, device=device),
                      torch.zeros(1, batch_size, decoder.rnn.hidden_size, device=device))
        input_tok = tgt[:,0]  # BOS
        loss = 0
        max_tgt = tgt.size(1)
        for t in range(1, max_tgt):
            preds, dec_hidden, attn = decoder(input_tok, dec_hidden, encoder_outputs, mask=src_mask)
            loss_step = criterion(preds, tgt[:,t])
            loss += loss_step
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = preds.argmax(1)
            input_tok = tgt[:,t] if teacher_force else top1
        loss = loss / (max_tgt - 1)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), clip)
        torch.nn.utils.clip_grad_norm_(decoder.parameters(), clip)
        enc_optimizer.step(); dec_optimizer.step()
        total_loss += loss.item() * batch_size
    return total_loss / len(data_loader.dataset)

def evaluate(encoder, decoder, data_loader, criterion):
    encoder.eval(); decoder.eval()
    total_loss = 0
    references = []
    hypotheses = []
    with torch.no_grad():
        for src, tgt, src_lens, tgt_lens in tqdm(data_loader):
            src, tgt = src.to(device), tgt.to(device)
            src_mask = make_src_mask(src, src_lens).to(device)
            encoder_outputs, enc_hidden = encoder(src, src_lens)
            batch_size = src.size(0)
            dec_hidden = (torch.zeros(1, batch_size, decoder.rnn.hidden_size, device=device),
                          torch.zeros(1, batch_size, decoder.rnn.hidden_size, device=device))
            input_tok = tgt[:,0]
            max_tgt = tgt.size(1)
            preds_seq = []
            for t in range(1, max_tgt):
                preds, dec_hidden, attn = decoder(input_tok, dec_hidden, encoder_outputs, mask=src_mask)
                loss_step = criterion(preds, tgt[:,t])
                total_loss += loss_step.item() * src.size(0)
                top1 = preds.argmax(1)
                preds_seq.append(top1.unsqueeze(1))
                input_tok = top1
            preds_seq = torch.cat(preds_seq, dim=1).cpu().numpy()
            # decode to text for BLEU
            for i in range(batch_size):
                hyp_ids = preds_seq[i].tolist()
                # remove after EOS if present
                if sp.piece_to_id(EOS) in hyp_ids:
                    hyp_ids = hyp_ids[:hyp_ids.index(sp.piece_to_id(EOS))]
                hypotheses.append(sp.decode(hyp_ids))
                tgt_ids = tgt[i].cpu().tolist()
                if sp.piece_to_id(EOS) in tgt_ids:
                    tgt_ids = tgt_ids[1:tgt_ids.index(sp.piece_to_id(EOS))]
                references.append(sp.decode(tgt_ids))
    avg_loss = total_loss / len(data_loader.dataset)
    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    return avg_loss, bleu.score, references[:5], hypotheses[:5]

In [ ]:
# Run experiments for General and Additive attention (tiny run, increase epochs for better results)
vocab_size = sp.get_piece_size()
emb_dim = 256
enc_hid = 256
dec_hid = 256
epochs = 3
results = {}
for attn_name in ['general', 'additive']:
    print('Running experiment with', attn_name, 'attention')
    if attn_name == 'general':
        attn_module = GeneralAttention(enc_hid, dec_hid)
    else:
        attn_module = AdditiveAttention(enc_hid, dec_hid)
    encoder = Encoder(vocab_size, emb_dim, enc_hid).to(device)
    decoder = Decoder(vocab_size, emb_dim, enc_hid, dec_hid, attn_module).to(device)
    enc_opt = torch.optim.Adam(encoder.parameters(), lr=0.001)
    dec_opt = torch.optim.Adam(decoder.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(ignore_index=sp.piece_to_id(PAD) if PAD in sp else 0)
    train_losses = []
    val_losses = []
    val_bleus = []
    for ep in range(1, epochs+1):
        tr_loss = train_epoch(encoder, decoder, train_loader, enc_opt, dec_opt, criterion)
        val_loss, val_bleu, refs, hyps = evaluate(encoder, decoder, val_loader, criterion)
        train_losses.append(tr_loss)
        val_losses.append(val_loss)
        val_bleus.append(val_bleu)
        print(f'Epoch {ep}: train_loss={tr_loss:.4f}, val_loss={val_loss:.4f}, val_BLEU={val_bleu:.2f}')
    results[attn_name] = dict(train_losses=train_losses, val_losses=val_losses, val_bleus=val_bleus, encoder=encoder, decoder=decoder)

# Save results summary
import pickle
with open('attention_results.pkl','wb') as f:
    pickle.dump(results, f)
print('Experiments finished and saved to attention_results.pkl')

In [ ]:
# Plot training/validation loss and BLEU for both experiments
import matplotlib.pyplot as plt
for k,v in results.items():
    plt.plot(v['train_losses'], label=f'{k}_train')
    plt.plot(v['val_losses'], label=f'{k}_val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

for k,v in results.items():
    plt.plot(v['val_bleus'], label=f'{k}_BLEU')
plt.xlabel('Epoch')
plt.ylabel('BLEU')
plt.legend()
plt.title('Validation BLEU scores')
plt.show()

## Attention maps visualization (Task 3.3)
We'll run a single inference on a sample sentence and display the attention weights heatmap for the selected model.

In [ ]:
# Visualize attention for a sample from validation set
sample_src, sample_tgt = next(iter(val_loader))[:2]
sample_src = sample_src[0:1].to(device)
sample_tgt = sample_tgt[0:1].to(device)
src_len = torch.tensor([sample_src.size(1)])
# choose additive model for visualization if available
model = results.get('additive', results.get('general'))
encoder = model['encoder']; decoder = model['decoder']
encoder.eval(); decoder.eval()
with torch.no_grad():
    enc_outs, _ = encoder(sample_src, src_len)
    dec_hidden = (torch.zeros(1,1,decoder.rnn.hidden_size, device=device), torch.zeros(1,1,decoder.rnn.hidden_size, device=device))
    input_tok = sample_tgt[:,0]  # BOS
    attn_weights = []
    for t in range(1, sample_tgt.size(1)):
        preds, dec_hidden, attn = decoder(input_tok, dec_hidden, enc_outs, mask=make_src_mask(sample_src, src_len).to(device))
        attn_weights.append(attn.cpu().numpy())
        input_tok = preds.argmax(1)
    attn_matrix = np.stack(attn_weights, axis=0).squeeze(1) if len(attn_weights)>0 else None

import numpy as np
if attn_matrix is not None:
    plt.figure(figsize=(8,6))
    sns.heatmap(attn_matrix, cmap='viridis')
    plt.xlabel('Source positions')
    plt.ylabel('Target positions')
    plt.title('Attention map')
    plt.show()
else:
    print('No attention weights captured')

## 3) Results analysis and saving to README (Task 3.4)
Brief analysis: compare final BLEU and loss curves between `general` and `additive` experiments and conclude which performed better on validation set. Save a short summary to `README_results.md`.

In [ ]:
# Build a small textual summary and save to README_results.md
summary_lines = ['Attention experiments summary:', '']
for k,v in results.items():
    summary_lines.append(f'{k} - final val loss: {v['val_losses'][-1]:.4f}, final BLEU: {v['val_bleus'][-1]:.2f}')
summary = '
'.join(summary_lines)
with open('README_results.md','w', encoding='utf-8') as f:
    f.write(summary)
print(summary)

## 4) Web application (Task 4)
Below is a minimal Flask app example to serve translations using the chosen best attention model. Save the code block to `app_translate.py` and run it. The app exposes `/translate` which accepts `POST` JSON with `text` field.

In [ ]:
flask_app_code = r
"""
from flask import Flask, request, jsonify
import torch
import sentencepiece as spm
# Load tokenizer and model artifact paths must be adjusted after training
sp = spm.SentencePieceProcessor()
sp.load('spm_en_ne.model')
# Load best model (this notebook saves `attention_results.pkl`)
import pickle
with open('attention_results.pkl','rb') as f:
    results = pickle.load(f)
# choose additive if present
model = results.get('additive', results.get('general'))
encoder = model['encoder']; decoder = model['decoder']
encoder.eval(); decoder.eval()
app = Flask(__name__)
@app.route('/translate', methods=['POST'])
def translate():
    data = request.get_json()
    text = data.get('text','')
    ids = sp.encode(text, out_type=int)
    src = torch.tensor([[sp.piece_to_id('<s>')] + ids + [sp.piece_to_id('</s>')]], dtype=torch.long)
    src_len = torch.tensor([src.size(1)])
    with torch.no_grad():
        enc_outs, _ = encoder(src.to(device), src_len.to(device))
        dec_hidden = (torch.zeros(1,1,decoder.rnn.hidden_size, device=device), torch.zeros(1,1,decoder.rnn.hidden_size, device=device))
        input_tok = torch.tensor([sp.piece_to_id('<s>')])
        output_ids = []
        for _ in range(80):
            preds, dec_hidden, attn = decoder(input_tok.to(device), dec_hidden, enc_outs, mask=(torch.arange(enc_outs.size(1)).unsqueeze(0)<src_len).to(device))
            top1 = preds.argmax(1).item()
            if top1 == sp.piece_to_id('</s>'):
                break
            output_ids.append(top1)
            input_tok = torch.tensor([top1])
    text_out = sp.decode(output_ids)
    return jsonify({'translation': text_out})
"""

# Write app file for the user to run
with open('app_translate.py','w', encoding='utf-8') as f:
    f.write(flask_app_code)
print('Wrote app_translate.py. Run with: python app_translate.py')

### Next steps and notes:
- Increase `epochs` and `vocab_size` for production-quality results.
- Add checkpointing and early stopping.
- For large datasets or better tokenization, experiment with separate tokenizers or pre-trained multilingual tokenizers.
- Attribution: Dataset credit to CohleM (Hugging Face). Tokenizer: SentencePiece (Google). Libraries: Hugging Face `datasets`, PyTorch, sacrebleu.